# Silver Layer — ICU Input Events (`inputevents`)

**Source:** `<source_schema>.inputevents` (default `cdr_mimiciv.bronze.inputevents`)
**Target:** `<target_schema>.inputevents` (default `cdr_mimiciv.silver.inputevents`)

**Module:** MIMIC-IV ICU (`icu/inputevents`, MetaVision). Links to `patients` on `subject_id`, `admissions` on `hadm_id`, `icustays` on `stay_id` and `d_items` on `itemid`.

**Objective:** Prepare ICU input (drug, fluid and nutrition administration) records for the Silver layer while preserving patient, admission and ICU stay identifiers, item identifiers, amounts, rates, units, order categories, statuses and event timestamps.

**Grain (official documentation):** One row per input event component, i.e. one item (`itemid`, e.g. norepinephrine or its NaCl solution) delivered between `starttime` and `endtime` at a given rate. A change in rate ends the current row and creates new rows. `orderid` groups the components of the same solution; `linkorderid` links the same order across rate changes. A bolus is listed as ending one minute after it started. The documentation defines no unique key, so uniqueness of any key is **not asserted**.

**Silver rules:** Preserve every source record, column and Bronze data type (pass-through). Never delete, filter, deduplicate, impute, aggregate or automatically correct clinical information. Reversed intervals (`endtime` < `starttime`) are reported and preserved, not corrected.

**Data quality checks encoded in this notebook (reviewed code; asserted only when `enforce_profiled_counts` is `true`):** 0 exact duplicate groups, 0 NULL identifiers (`subject_id`, `hadm_id`, `stay_id`, `itemid`, `orderid`), and 10 records with `endtime` earlier than `starttime`. Row preservation, schema preservation and Bronze–Silver agreement are always asserted.

**Publication:** The table is published with `CREATE OR REPLACE` (Delta) into `<target_schema>`; downstream Gold depends on it. Nothing is dropped.

**Cost note:** about 11M rows. Validation uses single-pass aggregates (no `exceptAll`, no repeated full scans).

**Dependencies:** none (reads Bronze only; no joins to other Silver tables).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

# Configure the inputevents source and target
SOURCE_TABLE = f"{BRONZE}.inputevents"
TARGET_TABLE = f"{SILVER}.inputevents"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)
print("Enforce profiled counts:", ENFORCE_PROFILED_COUNTS)

Source: cdr_mimiciv.bronze.inputevents
Target: cdr_mimiciv.silver.inputevents
Enforce profiled counts: True


In [0]:
# Load all ICU input events without modifying records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()

bronze_count = bronze_df.count()

print("Bronze rows:", bronze_count)
display(bronze_df.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- amountuom: string (nullable = true)
 |-- rate: double (nullable = true)
 |-- rateuom: string (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- linkorderid: integer (nullable = true)
 |-- ordercategoryname: string (nullable = true)
 |-- secondaryordercategoryname: string (nullable = true)
 |-- ordercomponenttypedescription: string (nullable = true)
 |-- ordercategorydescription: string (nullable = true)
 |-- patientweight: double (nullable = true)
 |-- totalamount: double (nullable = true)
 |-- totalamountuom: string (nullable = true)
 |-- isopenbag: integer (nullable = true)
 |-- continueinnextd

subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,amount,amountuom,rate,rateuom,orderid,linkorderid,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,ordercategorydescription,patientweight,totalamount,totalamountuom,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
17583408,23468047,31126544,36832,2145-06-22T23:00:00.000Z,2145-06-23T20:00:00.000Z,2145-06-23T02:06:00.000Z,225158,105.00000312924385,mL,5.0,mL/hour,2788905,2788905,02-Fluids (Crystalloids),Additive (Crystalloid),Main order parameter,Continuous IV,144.2,1000.0,mL,0,0,Paused,1000.0,5.0
17583408,23468047,31126544,36832,2145-06-22T23:30:00.000Z,2145-06-22T23:31:00.000Z,2145-06-23T02:06:00.000Z,220949,100.0,mL,null,null,3971496,3971496,08-Antibiotics (IV),02-Fluids (Crystalloids),Mixed solution,Drug Push,144.2,100.0,mL,0,0,FinishedRunning,100.0,0.0
17583408,23468047,31126544,36832,2145-06-22T23:30:00.000Z,2145-06-22T23:31:00.000Z,2145-06-23T02:06:00.000Z,225893,1.0,dose,null,null,3971496,3971496,08-Antibiotics (IV),02-Fluids (Crystalloids),Main order parameter,Drug Push,144.2,100.0,mL,0,0,FinishedRunning,1.0,1.0
17583408,23468047,31126544,36832,2145-06-23T00:50:00.000Z,2145-06-23T00:51:00.000Z,2145-06-23T02:07:00.000Z,220949,50.0,mL,null,null,8031865,8031865,08-Antibiotics (IV),02-Fluids (Crystalloids),Mixed solution,Drug Push,144.2,50.0,mL,0,0,FinishedRunning,50.0,0.0
17583408,23468047,31126544,36832,2145-06-23T00:50:00.000Z,2145-06-23T00:51:00.000Z,2145-06-23T02:07:00.000Z,225860,1.0,dose,null,null,8031865,8031865,08-Antibiotics (IV),02-Fluids (Crystalloids),Main order parameter,Drug Push,144.2,50.0,mL,0,0,FinishedRunning,1.0,0.0
17583408,23468047,31126544,36832,2145-06-23T22:00:00.000Z,2145-06-23T22:01:00.000Z,2145-06-24T05:14:00.000Z,226364,1000.0,mL,null,null,1674840,1674840,16-Pre Admission/Non-ICU,null,Main order parameter,Bolus,144.2,1000.0,mL,0,0,FinishedRunning,1000.0,1000.0
17583408,23468047,31126544,36832,2145-06-24T01:30:00.000Z,2145-06-24T02:06:00.000Z,2145-06-24T01:54:00.000Z,222168,77.9220741484074,mg,15.010417439043522,mcg/kg/min,3623026,3623026,01-Drips,02-Fluids (Crystalloids),Main order parameter,Continuous Med,144.2,100.0,mL,0,0,ChangeDose/Rate,1000.0,15.000000953674316
17583408,23468047,31126544,36832,2145-06-24T01:30:00.000Z,2145-06-24T02:06:00.000Z,2145-06-24T01:54:00.000Z,225943,7.792207717895508,mL,12.98701286315918,mL/hour,3623026,3623026,01-Drips,02-Fluids (Crystalloids),Mixed solution,Continuous Med,144.2,100.0,mL,0,0,ChangeDose/Rate,100.0,12.987000465393066
17583408,23468047,31126544,36832,2145-06-24T01:30:00.000Z,2145-06-25T22:12:00.000Z,2145-06-24T01:53:00.000Z,225158,223.50000666081905,mL,5.0,mL/hour,8196802,2788905,02-Fluids (Crystalloids),Additive (Crystalloid),Main order parameter,Continuous IV,144.2,1000.0,mL,0,0,Stopped,895.0,5.0
17583408,23468047,31126544,36832,2145-06-24T02:06:00.000Z,2145-06-24T02:34:00.000Z,2145-06-24T02:23:00.000Z,222168,40.404038447322364,mg,10.00694464892149,mcg/kg/min,1267018,3623026,01-Drips,02-Fluids (Crystalloids),Main order parameter,Continuous Med,144.2,100.0,mL,0,0,ChangeDose/Rate,922.0779418945312,10.000000953674316


In [0]:
# Inspect all Bronze columns and existing data types
schema_df = spark.createDataFrame(
    [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in bronze_df.schema.fields
    ],
    ["column_name", "bronze_type", "nullable"]
)

display(schema_df)

column_name,bronze_type,nullable
subject_id,int,true
hadm_id,int,true
stay_id,int,true
caregiver_id,int,true
starttime,timestamp,true
endtime,timestamp,true
storetime,timestamp,true
itemid,int,true
amount,double,true
amountuom,string,true


In [0]:
# Preserve all source records, columns and data types
silver_df = bronze_df

silver_df.printSchema()

# Ordered expected schema (pass-through of Bronze)
EXPECTED_SCHEMA = {name: dtype for name, dtype in silver_df.dtypes}

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- amountuom: string (nullable = true)
 |-- rate: double (nullable = true)
 |-- rateuom: string (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- linkorderid: integer (nullable = true)
 |-- ordercategoryname: string (nullable = true)
 |-- secondaryordercategoryname: string (nullable = true)
 |-- ordercomponenttypedescription: string (nullable = true)
 |-- ordercategorydescription: string (nullable = true)
 |-- patientweight: double (nullable = true)
 |-- totalamount: double (nullable = true)
 |-- totalamountuom: string (nullable = true)
 |-- isopenbag: integer (nullable = true)
 |-- continueinnextd

In [0]:
# Single pass: missingness for every column, identifier nulls and reversed intervals
identifier_columns = [
    c for c in ["subject_id", "hadm_id", "stay_id", "itemid", "orderid"]
    if c in silver_df.columns
]

has_interval = {"starttime", "endtime"}.issubset(silver_df.columns)
reversed_cond = (
    F.col("starttime").isNotNull() &
    F.col("endtime").isNotNull() &
    (F.col("endtime") < F.col("starttime"))
) if has_interval else F.lit(False)

profile = silver_df.agg(
    F.count("*").alias("total_records"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in silver_df.columns],
    F.sum(reversed_cond.cast("long")).alias("reversed_intervals")
).first().asDict()

display(spark.createDataFrame([(k, int(v or 0)) for k, v in profile.items()], ["metric", "value"]))

silver_count = profile["total_records"]
identifier_nulls = {f"null_{c}": profile[f"null_{c}"] for c in identifier_columns}
reversed_interval_count = int(profile["reversed_intervals"] or 0)

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Identifier NULL counts:", identifier_nulls)
print("Events with endtime before starttime (preserved, review only):", reversed_interval_count)

metric,value
total_records,10953713
null_subject_id,0
null_hadm_id,0
null_stay_id,0
null_caregiver_id,0
null_starttime,0
null_endtime,0
null_storetime,0
null_itemid,0
null_amount,0


Bronze rows: 10953713
Silver rows: 10953713
Identifier NULL counts: {'null_subject_id': 0, 'null_hadm_id': 0, 'null_stay_id': 0, 'null_itemid': 0, 'null_orderid': 0}
Events with endtime before starttime (preserved, review only): 10


In [0]:
# Identify exact duplicate groups without removing records (single pass)
dup_row = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("records_in_duplicate_groups"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("additional_duplicate_occurrences"),
        F.coalesce(F.max("count"), F.lit(0)).alias("maximum_group_size")
    )
    .first()
)
duplicate_count = dup_row["duplicate_groups"]

print("Exact duplicate inputevents groups:", duplicate_count)
print("Records in duplicate groups:", dup_row["records_in_duplicate_groups"])
print("Additional duplicate occurrences:", dup_row["additional_duplicate_occurrences"])
print("Maximum group size:", dup_row["maximum_group_size"])

Exact duplicate inputevents groups: 0
Records in duplicate groups: 0
Additional duplicate occurrences: 0
Maximum group size: 0


In [0]:
# Pre-publish validation: row and schema preservation always; profiled counts only when enforced
assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"
assert bronze_df.schema == silver_df.schema, "Bronze–Silver schema mismatch"

if ENFORCE_PROFILED_COUNTS:
    assert duplicate_count == 0, "Unexpected exact duplicate groups"
    assert all(v == 0 for v in identifier_nulls.values()), (
        f"Missing critical identifiers require review: {identifier_nulls}"
    )
    assert reversed_interval_count == 10, (
        "Source timestamp anomaly count changed; review before publication"
    )

print("All pre-publish checks passed.")

All pre-publish checks passed.


In [0]:
%sql
drop table cdr_mimiciv.silver.inputevents;

In [0]:
# Publish all inputevents records (CREATE OR REPLACE), preserving source timestamp anomalies
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)
print("Preserved reversed timestamp intervals:", reversed_interval_count)

Published: cdr_mimiciv.silver.inputevents
Preserved reversed timestamp intervals: 10


In [0]:
# Verify published inputevents records and schema (single aggregate pass)
published_df = spark.table(TARGET_TABLE)

published_schema = {name: dtype for name, dtype in published_df.dtypes}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from expected: {published_schema}"
)

published_stats = published_df.agg(
    F.count("*").alias("rows"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in identifier_columns],
    F.sum(reversed_cond.cast("long")).alias("reversed_intervals")
).first().asDict()

print("Bronze rows:", bronze_count)
print("Published rows:", published_stats["rows"])

assert published_stats["rows"] == bronze_count, "Published row-count mismatch"
assert int(published_stats["reversed_intervals"] or 0) == reversed_interval_count, (
    "Published reversed-interval count differs from Bronze"
)
assert {k: v for k, v in published_stats.items() if k.startswith("null_")} == identifier_nulls, (
    "Published identifier NULL counts differ from Bronze"
)

print("Silver inputevents table verified successfully.")

Bronze rows: 10953713
Published rows: 10953713
Silver inputevents table verified successfully.


In [0]:
dbutils.notebook.exit("inputevents:OK")